# Convexity and Convergence Guarantees

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/03_convexity_and_convergence.ipynb)

---

## 🎯 Learning Objective

Define convex functions and sets, and understand what convergence guarantees convexity buys an optimizer.

---

## 📐 Theory

`04.01` and `04.02` treated non-convexity as the default. This notebook studies the special case
where it's absent — because when a loss *is* convex, gradient descent's behavior becomes
provably predictable rather than merely empirically reasonable.

### Convex sets and functions

A set $C$ is **convex** if the line segment between any two points in $C$ stays inside $C$:
$\theta \mathbf{x}+(1-\theta)\mathbf{y} \in C$ for all $\mathbf{x},\mathbf{y}\in C$,
$\theta\in[0,1]$. A function $f$ is **convex** if its epigraph is convex, equivalently if it
lies below every chord connecting two of its points:

$$f(\theta \mathbf{x}+(1-\theta)\mathbf{y}) \le \theta f(\mathbf{x}) + (1-\theta) f(\mathbf{y})
\quad \forall\, \mathbf{x},\mathbf{y},\ \theta\in[0,1]$$

For twice-differentiable $f$, this is equivalent to the Hessian (`02.06`) being positive
semi-definite everywhere: $H(\mathbf{x}) \succeq 0\ \forall \mathbf{x}$. The payoff is enormous:
a convex function's **every local minimum is global**, and there are no spurious local minima
or saddle points to worry about — the exact opposite of `04.01`'s landscape.

### Strong convexity: a quantitative lower bound on curvature

Convexity says the Hessian's eigenvalues are $\ge 0$; **$\mu$-strong convexity** demands they're
bounded *away* from zero: $H(\mathbf{x}) \succeq \mu I$ for some $\mu>0$. Geometrically, $f$
curves upward at least as fast as the bowl $\frac{\mu}{2}\|\mathbf{x}\|^2$ in every direction.
This rules out the flat-bottomed case ($f(x)=x^4$ is convex but not strongly convex — its
Hessian is $0$ at the origin) and, crucially, guarantees a *unique* minimizer.

### Smoothness (Lipschitz gradients): an upper bound on curvature

The complementary condition, **$L$-smoothness**, bounds curvature from above:
$\|\nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\| \le L\|\mathbf{x}-\mathbf{y}\|$ — the gradient
can't change arbitrarily fast, equivalently $H(\mathbf{x}) \preceq LI$. Together, $\mu$-strong
convexity and $L$-smoothness sandwich every Hessian eigenvalue in $[\mu, L]$.

### Convergence rate: the condition number decides everything

For an $L$-smooth, $\mu$-strongly convex $f$, gradient descent with the optimal constant step
size $\eta=\frac{2}{L+\mu}$ converges **linearly**: $f(\mathbf{w}_k)-f^* \le \rho^k
\left(f(\mathbf{w}_0)-f^*\right)$ with rate $\rho=\left(\frac{L-\mu}{L+\mu}\right)^2$ (Nesterov,
2004; Boyd & Vandenberghe, 2004). Writing $\kappa=L/\mu$ (the **condition number** — recall
`01.06`'s connection between eigenvalue spread and matrix conditioning), $\rho$ approaches $1$
as $\kappa\to\infty$: an ill-conditioned bowl (elongated, like `04.02`'s correlated-feature
example) forces painfully slow convergence, while $\kappa\approx 1$ (nearly spherical contours)
converges almost immediately. This single number is *the* quantity that separates "easy" convex
optimization from "hard" convex optimization — non-convexity isn't the only source of
difficulty.

### Logistic regression: convex, in practice

Linear/logistic regression losses are convex (verified below), so training one has none of
`04.01`'s landscape hazards: gradient descent provably finds *the* global minimum, given enough
steps and a small enough $\eta$. Deep networks lose this guarantee entirely — their loss is a
composition of convex pieces through non-convex nonlinearities — which is precisely why Module
`04` needs the extra machinery (momentum, adaptive rates, second-order methods) that a convex
problem never requires.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The chord test makes convexity visible directly, and plotting gradient descent's theoretical
convergence rate against the condition number $\kappa$ shows exactly why some convex problems
are still much harder than others.

In [ ]:
# Left two panels: the chord test (definition of convexity) applied to a convex and a
# non-convex function. Right panel: gradient descent's theoretical linear-convergence rate
# rho = ((kappa-1)/(kappa+1))^2 as a function of the condition number kappa=L/mu.
def f_convex(x):
    return x ** 2

def f_nonconvex(x):
    return -2 * x**2 + x**4

x_range = np.linspace(-1.8, 1.8, 200)
x1, x2 = -1.5, 1.2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, f, name in [(axes[0], f_convex, "Convex: f(x)=x^2"),
                     (axes[1], f_nonconvex, "Non-convex: f(x)=-2x^2+x^4")]:
    ax.plot(x_range, f(x_range), color="#4C72B0", lw=2)
    chord_x = np.linspace(x1, x2, 50)
    chord_y = f(x1) + (f(x2) - f(x1)) * (chord_x - x1) / (x2 - x1)
    ax.plot(chord_x, chord_y, "--", color="#C44E52", lw=1.5, label="chord")
    ax.plot([x1, x2], [f(x1), f(x2)], "ko")
    below_chord = np.all(f(chord_x) <= chord_y + 1e-9)
    ax.set_title(f"{name}\nf always <= chord: {below_chord}")
    ax.legend(fontsize=8)

kappas = np.logspace(0, 3, 200)
rhos = ((kappas - 1) / (kappas + 1)) ** 2
axes[2].semilogx(kappas, rhos, color="#55A868", lw=2)
axes[2].set_xlabel("condition number kappa = L/mu"); axes[2].set_ylabel("convergence rate rho")
axes[2].set_title("Gradient descent's rate vs conditioning:\nkappa near 1 is fast, large kappa stalls")
plt.tight_layout()
plt.show()

print("Chord-test violation counts over 1000 random point pairs:")
rng = np.random.default_rng(0)
for f, name in [(f_convex, "f(x)=x^2"), (f_nonconvex, "f(x)=-2x^2+x^4")]:
    violations = 0
    for _ in range(1000):
        a, b = rng.uniform(-1.5, 1.5, 2)
        theta = rng.uniform(0, 1)
        if f(theta * a + (1 - theta) * b) > theta * f(a) + (1 - theta) * f(b) + 1e-9:
            violations += 1
    print(f"  {name}: {violations}/1000 violations")

## 🐍 Implementation from Scratch

We run gradient descent on quadratics with different condition numbers $\kappa$, measure the
*empirical* per-step shrinkage factor of the optimality gap, and check it against the
theoretical rate $\rho=\left(\frac{\kappa-1}{\kappa+1}\right)^2$ from the Theory section. Then we
verify logistic regression's loss is convex by checking its Hessian is positive semi-definite
across many random points, cross-checked against `scipy.optimize`.

In [ ]:
from scipy.optimize import minimize

def gd_on_quadratic(mu, L, steps=200):
    """Gradient descent on f(x)=0.5*x^T diag(mu,L) x, with the optimal constant step size
    eta=2/(L+mu) from the Theory section. Returns the optimality gap f(x_k)-f* at every step."""
    A = np.diag([mu, L])
    x = np.array([1.0, 1.0])
    eta = 2.0 / (L + mu)
    gaps = []
    for _ in range(steps):
        gaps.append(0.5 * x @ A @ x)   # f* = 0 here, so the gap IS the loss
        x = x - eta * (A @ x)
    return np.array(gaps)

print("Empirical vs theoretical convergence rate rho, for several condition numbers:")
for mu, L in [(1, 1), (1, 4), (1, 10), (1, 50)]:
    gaps = gd_on_quadratic(mu, L)
    kappa = L / mu
    theoretical_rho = ((kappa - 1) / (kappa + 1)) ** 2
    empirical_rho = (gaps[100:150] / gaps[99:149]).mean() if kappa > 1 else float("nan")
    print(f"  kappa={kappa:5.1f}:  theoretical rho={theoretical_rho:.5f}   empirical rho={empirical_rho:.5f}")

# ---------- Verify logistic regression's loss is convex ----------
rng = np.random.default_rng(0)
N, d = 200, 3
X = rng.normal(size=(N, d))
true_w = np.array([1.5, -2.0, 0.5])
y = (rng.uniform(size=N) < 1 / (1 + np.exp(-X @ true_w))).astype(float)

def logistic_loss(w, X, y):
    z = X @ w
    return np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)  # stable log(1+e^z)-yz

def logistic_grad(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return X.T @ (p - y) / len(y)

def logistic_hessian(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return (X * (p * (1 - p))[:, None]).T @ X / len(y)  # X^T diag(p(1-p)) X, always PSD

min_eigval = min(np.linalg.eigvalsh(logistic_hessian(rng.normal(size=d) * 2, X, y)).min() for _ in range(200))
print(f"\nSmallest Hessian eigenvalue over 200 random weight vectors: {min_eigval:.6f} (>= 0 means convex)")

# Cross-check our gradient descent solution against scipy's independent BFGS solver
w_gd = np.zeros(d)
for _ in range(2000):
    w_gd = w_gd - 0.5 * logistic_grad(w_gd, X, y)
result = minimize(logistic_loss, np.zeros(d), args=(X, y), jac=logistic_grad, method="BFGS")
print(f"our gradient descent: w={np.round(w_gd, 4)}, loss={logistic_loss(w_gd, X, y):.5f}")
print(f"scipy BFGS:           w={np.round(result.x, 4)}, loss={result.fun:.5f}")
print(f"match: {np.allclose(w_gd, result.x, atol=1e-3)}")

## 🤖 Why This Matters for AI

Logistic regression is the textbook example of convex ML: its loss (cross-entropy) has a
positive-semi-definite Hessian everywhere, so gradient descent provably converges to *the*
global minimum regardless of where it starts. Swap the linear model for even a tiny neural
network and that guarantee vanishes — the composition of a convex loss with a non-convex
feature map (the hidden layer) is generally non-convex, so different random initializations can
land in different, non-identical solutions, exactly as `04.01` predicted. We make that contrast
concrete below: run both models from many wildly different random starting points and compare
how much the final solutions actually differ.

In [ ]:
# Same classification task, two models: logistic regression (convex) vs a small 1-hidden-layer
# network (non-convex, since a tanh hidden layer composed with a convex loss is not convex).
# A touch of nonlinearity in the true decision boundary means neither model fits perfectly --
# what we care about is how much the LEARNED solution varies across random restarts.
rng_ai = np.random.default_rng(7)
N_ai, d_ai = 200, 3
X_ai = rng_ai.normal(size=(N_ai, d_ai))
z_true = X_ai @ np.array([1.5, -2.0, 0.5]) + 0.3 * np.sin(3 * X_ai[:, 0])
y_ai = (rng_ai.uniform(size=N_ai) < 1 / (1 + np.exp(-z_true))).astype(float)

def logistic_loss_ai(w, X, y):
    z = X @ w
    return np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)

def logistic_grad_ai(w, X, y):
    p = 1 / (1 + np.exp(-X @ w))
    return X.T @ (p - y) / len(y)

def mlp_loss_grad(w_flat, X, y, d_hidden):
    i1 = d_ai * d_hidden
    W1 = w_flat[:i1].reshape(d_ai, d_hidden)
    b1 = w_flat[i1:i1 + d_hidden]
    W2 = w_flat[i1 + d_hidden:i1 + 2 * d_hidden].reshape(d_hidden, 1)
    b2 = w_flat[-1]
    A1 = np.tanh(X @ W1 + b1)
    z = (A1 @ W2 + b2).ravel()
    p = 1 / (1 + np.exp(-z))
    loss = np.mean(np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0) - y * z)
    n = len(y)
    dz = (p - y) / n
    dW2 = A1.T @ dz[:, None]
    db2 = dz.sum()
    dZ1 = (dz[:, None] @ W2.reshape(1, -1)) * (1 - A1 ** 2)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return loss, np.concatenate([dW1.ravel(), db1.ravel(), dW2.ravel(), [db2]])

d_hidden = 6
n_params = d_ai * d_hidden + d_hidden + d_hidden + 1
n_restarts = 8

logistic_finals, mlp_finals = [], []
for seed in range(n_restarts):
    r = np.random.default_rng(seed)
    w_log = r.normal(scale=3.0, size=d_ai)
    for _ in range(1500):
        w_log = w_log - 0.3 * logistic_grad_ai(w_log, X_ai, y_ai)
    logistic_finals.append(logistic_loss_ai(w_log, X_ai, y_ai))

    w_mlp = r.normal(scale=3.0, size=n_params)
    for _ in range(1500):
        loss, grad = mlp_loss_grad(w_mlp, X_ai, y_ai, d_hidden)
        w_mlp = w_mlp - 0.3 * grad
    mlp_finals.append(loss)

print(f"{'model':22s} {'min loss':>9} {'max loss':>9} {'spread':>8}")
print(f"{'logistic (convex)':22s} {min(logistic_finals):9.6f} {max(logistic_finals):9.6f} "
      f"{max(logistic_finals)-min(logistic_finals):8.6f}")
print(f"{'1-hidden MLP (not)':22s} {min(mlp_finals):9.6f} {max(mlp_finals):9.6f} "
      f"{max(mlp_finals)-min(mlp_finals):8.6f}")
print(f"\n{n_restarts} wildly different random restarts: logistic regression lands on the SAME")
print("solution every time (spread is numerical noise); the MLP lands on genuinely different")
print("solutions -- direct evidence of convexity's 'every local min is global' guarantee at work.")

## 🏋️ Exercises

### Warm-up
1. For $f(x)=x^2$, $x_1=-2$, $x_2=3$, $\theta=0.4$, compute both sides of the chord-test
   inequality by hand and confirm it holds. Then repeat for $f(x)=-2x^2+x^4$ at the same points
   and confirm it fails.

### Practice
2. In `gd_on_quadratic`, add `mu=1, L=200` to the sweep and predict the theoretical `rho` by
   hand before running it. Then explain, in terms of `04.02`'s wander-ratio experiment, why an
   even larger `L` would make *stochastic* gradient descent on the same problem even harder to
   tune than the batch case shown here.

### Challenge
3. Change `d_hidden` in the AI section from 6 to 1 (barely any nonlinearity) and rerun the
   8-restart experiment. Report the new spread for the MLP and compare it to the logistic
   regression spread. Explain why a 1-hidden-unit network's non-convexity is much "milder" than
   a 6-unit one's, connecting back to `04.01`'s overparameterization discussion.

---

## 📚 Further Reading
- Boyd & Vandenberghe, [*Convex Optimization*](https://web.stanford.edu/~boyd/cvxbook/), Ch. 2–3
- Nesterov, *Introductory Lectures on Convex Optimization: A Basic Course* (2004), Ch. 2
- Bubeck, ["Convex Optimization: Algorithms and Complexity"](https://arxiv.org/abs/1405.4980)

---

*Next notebook: [Momentum and Adaptive Methods](04_momentum_and_adaptive_methods.ipynb)*